### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="california_house_prices_2020",
    dataset_year="2021",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/california-house-prices/data?select=train.csv",
    download_description="""
We utilize the train.csv from the Kaggle competition. We use the following commands to download and organize the data:

kaggle competitions download -c california-house-prices -f train.csv && unzip train.csv.zip train.csv && rm train.csv.zip
mkdir -p local-data-warehouse/california_house_prices_2020 && mv train.csv local-data-warehouse/california_house_prices_2020
""",
    # References
    academic_reference_bibtex=r"""@misc{d2lcourse2021california_house_prices,
  author       = {d2lcourse},
  title        = {California House Prices},
  year         = {2021},
  howpublished = {\url{https://kaggle.com/competitions/california-house-prices}},
  note         = {Kaggle}
}
""",
    academic_reference_bibtex_key="d2lcourse2021california_house_prices",
    license="Custom Kaggle Competition Rules",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
We follow some common preprocessing steps from Kaggle and enhance features as much as possible.

We found out that the data was web scraped from redfin.com and that the houses are ordered by the time they were sold such that a house with a higher ID was sold later. We confirmed this by looking at the history of various houses on redfin.com that we found in the dataset and checking their order in the ID with the time they were sold. Houses with a higher ID were sold later.

- We log scale the target variable.
- We do not have the exact dates but we know a higher ID means a later sale. We name the ID column accordingly to "time_index" and use it as a time feature.
- The descriptions did contain the last sold price with a standard phrase such as "This home last sold for $X in January 2020. The Zestimate for this house is $Y The Rent Zestimate for this home is $Z/mo.". In these cases $X is identical to the target variable. Given such a drastic data leakage, we remove all rows (5413) that contain this phrase in the description to ensure they are not different in other ways too.
- Note, sometime descriptions are missing and we intend that models/pipelines need to be able to handle this.
- When investigating the "Lot" column we found many cases where the lot size (given in sq ft) is incorrect compared to the official lot size on the internet. The parser seems to have had an issue because the website often incorrectly showed the sq ft but gave the unit as acres. We found cases that were wrong by checking the acres size of the houses and anything with more than 2000 acres was investigated and subsequently corrected.
- We parsed the bedrooms with a proxy following Kaggle and added an additional column containing only the semi-free text descriptions of the bedrooms.
- We fix the parser errors for two cases for total interior livable area that were clear outliers based on the real data on redfin.
- Garage and total space were in most cases the same values parsed from different fields. We keep Garage space and add an 'Extra Space' column that is the difference between total and garage space as an additional feature. This results in some cases with negative extra space that are most likely parser errors. We set them to nan. Note, we observed that that garage space number is in many cases a parsing error and does not reflect the real garage count.
- The data has a lot of spatial features that are already semi-resolved to distances. This could be further enhanced using the address information. We leave it to the pipeline to handle this, if at all.
- The dataset contains many text-like many categorical that are also high-cardinality, so perfect string data.
- We drop all entries from the State of Arizona (AZ) as they are only a small fraction (n=431 after our preprocessing). The dataset is specified to be for California house prices.
- We treat ZIP codes as strings. But we add a new categorical features that puts zip code in to approximate relevant buckets of regions.
- We fixed one entry that had a negative value for garage space, likely due to a typo. This error still exists on the website.
- After all the above preprocessing, there exist 63 houses that appear multiple times in the dataset (same address, zip code, year built). We only keep the newest entry to avoid group-related target leakage.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Sold Price",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="time_index",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

df = df.rename(columns={
    "Id": "time_index",
})

as_date_cols = ["Listed On", "Last Sold On"]
as_string_cols = [
    "Summary", "Type", "Heating", "Cooling", "Parking", "Region",
    "Elementary School", "Middle School", "High School", "Flooring",
    "Heating features", "Cooling features", "Appliances included",
    "Laundry features", "Parking features", "City",
    "Address", "State",
]
as_cat_type = ["Type"]

# Remove rows with target leakage in description
leak_mask = ~df["Summary"].isna() & (df["Summary"].str.contains("last sold for"))
df = df[~leak_mask]

# Set wrongly parsed years to NaN
df.loc[df["Year built"].isin([0, 19, 9999, 1471, 2281]), "Year built"] = np.nan


# Fix wrong lot sizes
#   - found by: wrong_lot_size_mask = (df["Lot"]/43560) > 2000
for time_index, correct_size in [
    (1701, 6660), (5198, 5500), (11205, 5768), (16036, 4200),
    (17241, 7650), (19469, 60984), (23605, 7700), (23807, 35283.6),
    (30942, 4500), (31871, 6500), (34622, 6403), (39769, 2584), (43009, 7546),
    # still says acres
    (13280, 5804),(46033, 6216),
]:
    df.loc[df["time_index"] == time_index, "Lot"] = correct_size
assert ((df["Lot"]/43560) > 2000).sum() == 0, "There are still wrong lot sizes > 2000 acres"

# Ensure dtypes
c = "Year built" # special case
nan_mask = df[c].isna()
df.loc[nan_mask, c] = df.loc[~df[c].isna(), c].iloc[0]
df[c] = pd.to_datetime(df[c].astype(int).astype(str), errors='raise')
df.loc[nan_mask, c] = pd.NaT

for c in as_date_cols:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = df.loc[~df[c].isna(), c].iloc[0]  # Temp fill to allow conversion
    df[c] = pd.to_datetime(df[c], errors='raise')
    df.loc[nan_mask, c] = pd.NaT


c = "Zip" # ensure zip string is without .0 at the end.
nan_mask = df[c].isna()
df[c] = df[c].astype("string")
df.loc[nan_mask, c] = np.nan
df.loc[~nan_mask, c] = df.loc[~nan_mask, c].astype(int).astype("string")

for c in as_string_cols:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

# Extra FE for Bedrooms
bedroom_description_mask = pd.to_numeric(df["Bedrooms"], errors="coerce").isna()
df["Bedrooms_description"] = df["Bedrooms"].astype("string")
df.loc[(~bedroom_description_mask) | (df["Bedrooms"].isna()), "Bedrooms_description"] = np.nan
df["Bedrooms_description"] = df["Bedrooms_description"].astype("string")

# Compute a proxy for bedrooms from the description
# https://www.kaggle.com/code/wuwawa/automl-using-h2o
def bedroom_proxy(x):
    if not pd.isna(x) and not x.isdigit():
        temp = x.split(',')
        n = len(x.split(','))
        if 'Walk-in Closet' in temp:
            n -= 1
        if "Reverse Floor Plan" in temp:
            n -= 1
        if n <= 0:
            n = np.nan
        return n
    else:
        return x
df.loc[bedroom_description_mask, "Bedrooms"] = df.loc[bedroom_description_mask, "Bedrooms"].apply(bedroom_proxy)
df["Bedrooms"] = pd.to_numeric(df["Bedrooms"], errors="coerce")

# Fix wrong sqft parsing outliers
for time_index, correct_size in [
    (43911, 702),
    (26533, 700),
]:
    df.loc[df["time_index"] == time_index, "Total interior livable area"] = correct_size

# Add extra space and remove total space duplicates
df.loc[df["time_index"] == 41450, "Garage spaces"] = 5 # fix data error
df["Extra Space"] = df["Total spaces"] - df["Garage spaces"]
df.loc[df["Extra Space"] < 0, "Extra Space"] = np.nan
df = df.drop(columns=["Total spaces"])

# Drop entries from AZ state
df = df[df["State"] != "AZ"]
df = df.drop(columns=["State"])

# Zip code buckets for CA
def zip_buckets(x):
    if pd.isna(x):
        return np.nan

    x = int(x)

    if x < 91911:
        return "Southern California / LA Area"

    if x < 93000:
        return "San Diego Area"
    if x < 94000:
        return "Central California"
    if x < 95200:
        return "San Francisco Bay Area"
    if x < 97000:
        return "Northern California"

    raise ValueError(f"Unexpected zip code: {x}")

df["Zip Region"] = df["Zip"].apply(zip_buckets).astype("category")

# Only keep the newest entry for houses that appear multiple times
n_data = len(df)
idx = df.groupby(["Address", "Zip", "Year built"], dropna=False)["time_index"].idxmax()
df = df.loc[idx].reset_index(drop=True)
assert n_data - len(df) == 67 # We have 63 houses with multiple entries (either 2 or 3) for a total of 130 entire. We keep one entry per house, thus 130 - 63 = 67

# Log scale target as it is exp distributed
df[task_mold.target_column_name] = np.log1p(df[task_mold.target_column_name])

# Rest time_index (to avoid gaps from dropping rows)
df = df.drop(columns=["time_index"]).reset_index().rename(columns={"index": "time_index"})

Loaded data shape: (47439, 41)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 41,528
Columns: 42
Use sampling: False (sample size: 41,528)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['time_index', 'Address', 'Summary', 'Tax assessed value', 'Annual tax amount', 'Lot', 'Last Sold On', 'Parking', 'Parking features', 'Appliances included']
Rows remaining as candidates after top-10 filter: 0 (of 41,528)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,time_index,Address,Sold Price,Summary,Type,Year built,Heating,Cooling,Parking,Lot,Bedrooms,Bathrooms,Full bathrooms,Total interior livable area,Garage spaces,Region,Elementary School,Elementary School Score,Elementary School Distance,Middle School,Middle School Score,Middle School Distance,High School,High School Score,High School Distance,Flooring,Heating features,Cooling features,Appliances included,Laundry features,Parking features,Tax assessed value,Annual tax amount,Listed On,Listed Price,Last Sold On,Last Sold Price,City,Zip,Bedrooms_description,Extra Space,Zip Region
0,0,0 113th St,11.982935,"LAND LAND LAND FOR SALE! CALLING ALL INVESTORS WHO KNOW THIS AREA OF LOS ANGELES. LAND IS READY FOR A SFR TO BE CONSTRUCTED. BUILD AND FLIP! HOME NEXT DOOR IS IDENTICAL AND HAS 3 BEDS 2 BATHS. SEE TO APPRECIATE! LAND IS CEMENTED AND HAS ALLEY ACCESS. MAKE AN OFFER, THIS ONE WILL NOT LAST! NOTE: LATEST 1/30/2020* FOR FURTHER INFO ON HOW YOU CAN OBTAIN PERMITS FOR THIS LAND ANY LOS ANGELES (CITY) OFFICES HAVE ALL THE REQUIREMENTS FOR YOU TO USE THIS LAND TO BUILD. CITY HAS AN ADDRESS ADDRESS 1601 E 113TH ST LOS ANGELES. YOU CAN USE 45% OF THE LAND TO BUILD A HOME OF YOUR CHOICE. ZONE IS R1. CALL ME FOR FURTHER QUESTIONS...",VacantLand,NaT,<NA>,<NA>,0 spaces,NaN,NaN,NaN,NaN,NaN,NaN,Los Angeles,<NA>,NaN,NaN,Samuel Gompers Middle School,2.0,0.9,George Washington Preparatory High School,2.0,1.3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,2020-01-25,169999.0,NaT,NaN,Los Angeles,90044,<NA>,NaN,Southern California / LA Area
1,1,0 40 Ac Wilkins Rd,11.982935,"- Prime flat/level residential/agricultural lot - Easy access highway 111 - Canal access with water right - Paved road - Utilities are on or close by to the parcel - Excellent opportunity to build a farm - 40 acres of flat/level land - 1,742,400 sqft Interested buyers can also purchase together with parcel 003-120-007-000 (164 ac) and parcel 003-110-025-000 (17 ac) for a total of 221 acres.",VacantLand,NaT,Gas,<NA>,0 spaces,NaN,NaN,NaN,NaN,NaN,NaN,Niland,Grace Smith Elementary School,6.0,4.9,Bill E. Young Jr. Middle School,4.0,2.3,Calipatria High School,6.0,2.3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,2019-02-11,200000.0,NaT,NaN,Calipatria,92233,<NA>,NaN,San Diego Area
2,2,0 Acacia St,11.652696,<NA>,VacantLand,NaT,<NA>,<NA>,0 spaces,NaN,NaN,NaN,NaN,NaN,NaN,Delhi,Schendel Elementary School,5.0,0.2,Delhi Middle,4.0,0.8,Delhi High School,5.0,0.9,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,2020-06-17,120000.0,NaT,NaN,Delhi,95315,<NA>,NaN,Northern California
3,3,0 Apn 255 #280-24-00,11.571204,"Are you looking for a get away, then this might be place for you. 20 acres of land in the mountains of Lebec. Ready for you to build your new house or put a trailer on it. Property has water to it already and power is not to far from the property. Take a look at the views.",VacantLand,NaT,<NA>,<NA>,0 spaces,NaN,NaN,NaN,NaN,NaN,NaN,Lebec,Gorman Elementary School,3.0,5.2,<NA>,NaN,NaN,Quartz Hill High School,6.0,35.1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,2020-06-27,174900.0,NaT,NaN,Lebec,93243,<NA>,NaN,Central California
4,4,0 Avenue 71,13.652993,"Warm farmground. 15 acres medjools, 5 acres tall deglets, 6 acres young offshoot deglets. 50 acres open farmground. The following APNs are also apart of this 2 parcel property: 729-120-004 (39.45 ac)",VacantLand,NaT,<NA>,<NA>,0 spaces,NaN,NaN,NaN,NaN,NaN,NaN,Mecca,Saul Martinez Elementary School,3.0,3.1,Toro Canyon Middle School,2.0,8.3,Desert Mirage High School,3.0,8.3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,2017-11-10,2220000.0,NaT,NaN,Mecca,92254,<NA>,NaN,San Diego Area


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Zip Region,category,0.0,0.00,5.0,"San Francisco Bay Area, Southern California / LA Area, Northern California, Central California, San Diego Area"
1,Last Sold On,datetime64[ns],15536.0,37.41,5903.0,"2017-06-30 00:00:00, 2016-09-30 00:00:00, 2017-05-26 00:00:00, 2015-02-27 00:00:00, 2019-08-30 00:00:00, 2016-07-29 00:00:00, 2017-12-29 00:00:00, 2013-08-30 00:00:00, 2018-05-31 00:00:00, 2019-04-30 00:00:00"
2,Year built,datetime64[ns],1076.0,2.59,157.0,"1973-01-01 00:00:00, 2020-01-01 00:00:00, 1950-01-01 00:00:00, 1947-01-01 00:00:00, 1924-01-01 00:00:00, 1972-01-01 00:00:00, 1948-01-01 00:00:00, 1955-01-01 00:00:00, 1923-01-01 00:00:00, 1979-01-01 00:00:00"
3,Listed On,datetime64[ns],0.0,0.00,2266.0,"2020-09-18 00:00:00, 2020-10-16 00:00:00, 2020-10-02 00:00:00, 2020-10-09 00:00:00, 2020-08-14 00:00:00, 2020-08-07 00:00:00, 2020-09-25 00:00:00, 2020-10-23 00:00:00, 2020-09-11 00:00:00, 2020-10-01 00:00:00"
4,Last Sold Price,float64,15536.0,37.41,3723.0,"650000.0, 850000.0, 500000.0, 550000.0, 1100000.0, 1200000.0, 750000.0, 350000.0, 700000.0, 300000.0"
5,Middle School Score,float64,14458.0,34.82,9.0,"6.0, 3.0, 5.0, 7.0, 4.0, 8.0, 2.0, 9.0, 1.0"
6,Middle School Distance,float64,14457.0,34.81,212.0,"0.6, 0.8, 0.7, 0.5, 0.9, 0.4, 1.0, 0.3, 1.1, 1.2"
7,Lot,float64,13004.0,31.31,7903.0,"5000.0, 6098.0, 6534.0, 2500.0, 2495.0, 2996.0, 11325.6, 7405.0, 5998.0, 10890.0"
8,Full bathrooms,float64,6772.0,16.31,16.0,"2.0, 1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0"
9,High School Score,float64,4778.0,11.51,10.0,"7.0, 6.0, 8.0, 5.0, 4.0, 3.0, 9.0, 2.0, 10.0, 1.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
time_index,41528.0,2.076350e+04,1.198825e+04,0.000000,4.152700e+04
Sold Price,41528.0,1.377772e+01,7.755062e-01,11.517923,1.831532e+01
Lot,28524.0,5.405607e+04,7.977161e+05,0.000000,6.969600e+07
Bedrooms,37964.0,2.830497e+00,1.305326e+00,0.000000,4.200000e+01
Bathrooms,38568.0,2.356876e+00,1.196705e+00,0.000000,2.400000e+01
Full bathrooms,34756.0,2.087985e+00,9.803851e-01,1.000000,1.700000e+01
Total interior livable area,39221.0,1.830280e+03,1.187843e+03,1.000000,3.597000e+04
Garage spaces,40817.0,1.455643e+00,8.649698e+00,0.000000,1.000000e+03
Elementary School Score,37391.0,5.734508e+00,2.120630e+00,1.000000,1.000000e+01
Elementary School Distance,37519.0,1.071526e+00,2.235817e+00,0.000000,5.720000e+01


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column               rank                                                                        
Address              1                                                            421 Grove St   
                     2                                                            1529 6th Ave   
                     3                                          2476 Martin Luther King Jr Way   
                     4                                                540 Delancey St UNIT 204   
                     5                                                            627 16th Ave   
Appliances included  1                                                                    <NA>   
                     2     Dishwasher, Dryer, Garbage disposal, Microwave, Range / Oven, Re...   
                     3     Dishwasher, Dryer, Freezer, Garbage disposal, Microwave, Range /...   
                     4                                                           Dryer, Washer   
                     5                                                              Dishwasher   
Bedrooms_description 1                                                                    <NA>   
                     2                                                          Walk-in Closet   
                     3                                  Master Suite / Retreat, Walk-in Closet   
                     4                                                  Master Suite / Retreat   
                     5                                                    Ground Floor Bedroom   
City                 1                                                             Los Angeles   
                     2                                                                San Jose   
                     3                                                           San Francisco   
                     4                                                             Santa Clara   
                     5                                                               San Mateo   
Cooling              1                                                                    <NA>   
                     2                                                             Central Air   
                     3                                                              Central AC   
                     4                                                                 Central   
                     5                                                             Ceiling Fan   
Cooling features     1                                                                    <NA>   
                     2                                                                 Central   
                     3                                                             Central Air   
                     4                                                                   Other   
                     5                                                     Wall/Window Unit(s)   
Elementary School    1                                                                    <NA>   
                     2                                                Laurel Elementary School   
                     3                                               Sherman Elementary School   
                     4                                        Daniel Webster Elementary School   
                     5                                         Kenter Canyon Elementary School   
Flooring             1                                                                    <NA>   
                     2                                                                    Wood   
                     3                                                                Hardwood   
                     4                                                              Tile, Wood   
                     5                                                 

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.026,-0.205,0.601,0.003,log,574097.8,6.882298e+09,exponential


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import TimeSeriesSplit

spliter = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None)
splits = {0: {}, 1: {}, 2: {}}
for i, (train_idx, test_idx) in enumerate(spliter.split(df)):
    print(len(train_idx), len(test_idx))
    splits[i][0] = (train_idx, test_idx)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create a sklearn TimeSeriesSplit with 3 splits. We treat each split as a separate repeat with one fold. Each fold has 25% of the original data as test set. We simulate as if we deploy a model at a time point x, for the next time period (e.g. for a whole next year like in the original competition task). As we do not know the time periods from the time_index, we split based on number of samples.",
    splits=splits,
    time_horizon=10382,
    time_horizon_unit="steps",
)

10382 10382
20764 10382
31146 10382


## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d3064-3158-7e93-b010-51a7b6c72ba1
ad0d1b46cc651f864ac32cdb0cedd31f82596883e50eb11aef7dac83bc55f39c
